# Input

In [ ]:
import os
import sys
from pathlib import Path

# Thread limits should be set before importing torch / numpy-heavy libraries
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# Make local ProSST repo importable
PROSST_REPO = Path("/home/kebau102/Masterarbeit/ProSST")

if PROSST_REPO.exists():
    sys.path.insert(0, str(PROSST_REPO))
    print(f"Added ProSST repo to sys.path: {PROSST_REPO}")
else:
    print(f"[WARNING] ProSST repo not found: {PROSST_REPO}")

print("Environment setup done.")

Added ProSST repo to sys.path: /home/kebau102/Masterarbeit/ProSST
Environment setup done.


In [ ]:
import re
from dataclasses import dataclass, field
from typing import Iterable, Optional

import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm
print("Standard imports loaded.")

import torch
print("Torch loaded.")

from transformers import AutoModelForMaskedLM, AutoTokenizer
print("Transformers loaded.")

from prosst.structure.get_sst_seq import SSTPredictor
print("ProSST imports loaded.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Standard imports loaded.
Torch loaded.
Transformers loaded.
ProSST imports loaded.
Using device: cuda


# Path

In [ ]:
BASE_DIR = Path("/home/kebau102/Masterarbeit")
DATA_DIR = BASE_DIR / "data"

PDB_DIR = DATA_DIR / "pdb"
PETASE_PDB_DIR = DATA_DIR / "PETase_PDB"

OUT_DIR = BASE_DIR / "out" / "prosst" / "multi"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "AI4Protein/ProSST-2048"
STRUCTURE_VOCAB_SIZE = 2048

OFFSET_SEARCH = 200
STRICT_WT_MATCH = False

ALPHA_AMYLASE_PDB = PDB_DIR / "alpha_amylase_wt.pdb"

print(f"Output directory: {OUT_DIR}")
print(f"Alpha-Amylase PDB: {ALPHA_AMYLASE_PDB}")

Output directory: /home/kebau102/Masterarbeit/out/prosst/multi
Alpha-Amylase PDB: /home/kebau102/Masterarbeit/data/pdb/alpha_amylase_wt.pdb


In [ ]:
# Dataset Configuration Class
@dataclass
class DatasetConfig:
    name: str
    data_path: Path
    pdb_path: Path
    out_path: Path

    sequence_col: str = "sequence"
    variant_col: str = "variant"
    score_cols: list[str] = field(default_factory=lambda: ["score"])

    sep: Optional[str] = None
    require_num_mutations_col: bool = False
    require_sequence_length_match: bool = True

    offset_search: int = OFFSET_SEARCH
    strict_wt_match: bool = STRICT_WT_MATCH

In [ ]:
DATASETS = [
    DatasetConfig(
        name="UBE4B_multi",
        data_path=DATA_DIR / "ube4b_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "UBE4B.pdb",
        out_path=OUT_DIR / "UBE4B_prosst_multi.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="GRB2_multi",
        data_path=DATA_DIR / "grb2_with_full_sequence.tsv",
        pdb_path=PDB_DIR / "GRB2.pdb",
        out_path=OUT_DIR / "GRB2_prosst_multi.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="DLG4_abundance_multi",
        data_path=DATA_DIR / "dlg4-2022-abundance.tsv",
        pdb_path=PDB_DIR / "DLG4.pdb",
        out_path=OUT_DIR / "DLG4_abundance_prosst_multi.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),

    DatasetConfig(
        name="DLG4_binding_multi",
        data_path=DATA_DIR / "dlg4-2022-binding.tsv",
        pdb_path=PDB_DIR / "DLG4.pdb",
        out_path=OUT_DIR / "DLG4_binding_prosst_multi.tsv",
        sequence_col="sequence",
        variant_col="variant",
        score_cols=["score"],
        sep="\t",
        require_num_mutations_col=True,
        require_sequence_length_match=True,
    ),
]


for cfg in DATASETS:
    print(f"{cfg.name:24s} | data={cfg.data_path} | pdb={cfg.pdb_path}")

UBE4B_multi              | data=/home/kebau102/Masterarbeit/data/ube4b_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/UBE4B.pdb
GRB2_multi               | data=/home/kebau102/Masterarbeit/data/grb2_with_full_sequence.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/GRB2.pdb
DLG4_abundance_multi     | data=/home/kebau102/Masterarbeit/data/dlg4-2022-abundance.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/DLG4.pdb
DLG4_binding_multi       | data=/home/kebau102/Masterarbeit/data/dlg4-2022-binding.tsv | pdb=/home/kebau102/Masterarbeit/data/pdb/DLG4.pdb


In [ ]:
# Load ProSST Model
prosst_model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
).to(DEVICE)

prosst_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

prosst_model.eval()

sst_predictor = SSTPredictor(
    structure_vocab_size=STRUCTURE_VOCAB_SIZE
)

print("ProSST model, tokenizer and SST predictor loaded.")

---------- Load Model on cuda ----------
MODEL: 5.90M parameters
ProSST model, tokenizer and SST predictor loaded.


In [ ]:
# Variant Parsing
VAR_SPLIT = re.compile(r"[:;, \t]+")
VARIANT_PATTERN = re.compile(r"^([A-Z])(\d+)([A-Z])$")


def parse_variant(variant: object) -> list[tuple[str, int, str]]:
    """
    Parses variants like A123B.

    Multi-mutants can be separated by:
    ':', ';', ',', spaces or tabs.

    Returns:
        [(wild_type_residue, position_1based, mutant_residue)]
    """
    if pd.isna(variant):
        return []

    text = str(variant).strip().upper()

    if text in {"", "WT", "WILDTYPE", "WILD_TYPE"}:
        return []

    mutations = []

    for token in [p for p in VAR_SPLIT.split(text) if p]:
        match = VARIANT_PATTERN.match(token)

        if match is None:
            raise ValueError(
                f"Unrecognized variant token '{token}' in variant '{variant}'"
            )

        wt = match.group(1)
        pos = int(match.group(2))
        mt = match.group(3)

        mutations.append((wt, pos, mt))

    return mutations


def is_valid_variant(variant: object) -> bool:
    """
    True if the variant can be parsed and contains at least one mutation.
    This includes both single and multi mutants.
    """
    try:
        return len(parse_variant(variant)) >= 1
    except ValueError:
        return False


def count_mutations_from_variant(variant: object) -> int:
    """
    Returns the number of parsed mutations.
    Invalid variants return 0.
    """
    try:
        return len(parse_variant(variant))
    except ValueError:
        return 0

In [ ]:
def read_table(cfg: DatasetConfig) -> pd.DataFrame:
    """
    Reads CSV or TSV depending on cfg.sep.
    If cfg.sep is None, pandas tries to infer the separator.
    """
    if not cfg.data_path.exists():
        raise FileNotFoundError(f"Missing input table: {cfg.data_path}")

    if cfg.sep is None:
        return pd.read_csv(cfg.data_path, sep=None, engine="python")

    return pd.read_csv(cfg.data_path, sep=cfg.sep)


def save_table(df: pd.DataFrame, path: Path) -> None:
    """
    Saves as TSV if suffix is .tsv, otherwise CSV.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix.lower() == ".tsv":
        df.to_csv(path, sep="\t", index=False)
    else:
        df.to_csv(path, index=False)


def get_structure_from_pdb(pdb_path: Path) -> tuple[str, list[int]]:
    """
    Extracts amino acid sequence and ProSST structure tokens from a PDB file.
    """
    if not pdb_path.exists():
        raise FileNotFoundError(f"Missing PDB file: {pdb_path}")

    pred = sst_predictor.predict_from_pdb(str(pdb_path))[0]

    pdb_aa_seq = pred["aa_seq"].strip().upper()
    structure_sequence = pred[f"{STRUCTURE_VOCAB_SIZE}_sst_seq"]

    if len(pdb_aa_seq) != len(structure_sequence):
        raise ValueError(
            f"PDB aa_seq length ({len(pdb_aa_seq)}) differs from "
            f"SST length ({len(structure_sequence)}) for {pdb_path}"
        )

    return pdb_aa_seq, structure_sequence


def validate_sequence_length(
    wt_seq: str,
    pdb_seq: str,
    cfg: DatasetConfig,
) -> None:
    """
    Optional check: table WT sequence and PDB sequence should have same length.
    """
    if cfg.require_sequence_length_match and len(wt_seq) != len(pdb_seq):
        raise ValueError(
            f"{cfg.name}: WT sequence length ({len(wt_seq)}) differs from "
            f"PDB sequence length ({len(pdb_seq)})."
        )

In [ ]:
def build_ss_input_ids(structure_sequence: Iterable[int]) -> torch.Tensor:
    """
    ProSST expects:
    - start token: 1
    - structure tokens shifted by +3
    - end token: 2
    """
    structure_sequence_offset = [int(i) + 3 for i in structure_sequence]

    ss_input_ids = torch.tensor(
        [1, *structure_sequence_offset, 2],
        dtype=torch.long,
    ).unsqueeze(0)

    return ss_input_ids


def tokenize_reference_sequence(
    ref_seq: str,
    structure_sequence: Iterable[int],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Tokenizes the protein sequence and builds corresponding structure input IDs.
    """
    tokenized = prosst_tokenizer(
        [ref_seq],
        return_tensors="pt",
    )

    input_ids = tokenized["input_ids"].to(DEVICE)
    attention_mask = tokenized["attention_mask"].to(DEVICE)

    ss_input_ids = build_ss_input_ids(structure_sequence).to(DEVICE)

    if input_ids.shape[1] != ss_input_ids.shape[1]:
        raise ValueError(
            f"Tokenized sequence length ({input_ids.shape[1]}) differs from "
            f"structure token length ({ss_input_ids.shape[1]})."
        )

    return input_ids, attention_mask, ss_input_ids

In [ ]:
def find_best_offset(
    df: pd.DataFrame,
    ref_seq: str,
    variant_col: str,
    search: int = OFFSET_SEARCH,
) -> tuple[int, int, int]:
    """
    Finds the best offset d such that:

        ref_seq[pos1 + d - 1] == wild-type residue

    This version uses all mutations from both single and multi mutants.
    """
    parsed = []

    for variant in df[variant_col].tolist():
        muts = parse_variant(variant)

        for mut in muts:
            parsed.append(mut)

    best_offset = 0
    best_hits = -1
    best_total = 0

    n = len(ref_seq)

    for offset in range(-search, search + 1):
        hits = 0
        total = 0

        for wt, pos1, _ in parsed:
            pos_ref = pos1 + offset

            if 1 <= pos_ref <= n:
                total += 1

                if ref_seq[pos_ref - 1].upper() == wt:
                    hits += 1

        if hits > best_hits:
            best_offset = offset
            best_hits = hits
            best_total = total

    return best_offset, best_hits, best_total

In [ ]:
# ProSST Multi-Mutation Scoring
@torch.no_grad()
def score_variant(
    variant: object,
    ref_seq: str,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    ss_input_ids: torch.Tensor,
    offset: int,
    strict_wt_match: bool,
) -> tuple[float, bool, list[int], int]:
    """
    Scores one variant.

    For a single mutant:
        score = log P(mutant residue) - log P(wild-type residue)

    For a multi mutant:
        score = sum over all mutated positions of
                log P(mutant residue) - log P(wild-type residue)

    Each mutation position is masked individually on the same reference sequence.
    """
    muts = parse_variant(variant)

    if len(muts) == 0:
        return 0.0, True, [], 0

    total_score = 0.0
    all_wt_match = True
    ref_positions = []

    for wt, pos1, mt in muts:
        pos_ref = pos1 + offset

        if pos_ref < 1 or pos_ref > len(ref_seq):
            return float("nan"), False, ref_positions, len(muts)

        wt_ref = ref_seq[pos_ref - 1].upper()
        wt_matches = wt_ref == wt

        if not wt_matches:
            all_wt_match = False

        if strict_wt_match and not wt_matches:
            return float("nan"), False, ref_positions, len(muts)

        # If strict=False and WT does not match,
        # use the residue from the PDB/reference sequence as WT.
        wt_used = wt if wt_matches else wt_ref

        # CLS token is at index 0, residue 1 is at token index 1
        token_index = pos_ref

        masked_input_ids = input_ids.clone()
        masked_input_ids[0, token_index] = prosst_tokenizer.mask_token_id

        outputs = prosst_model(
            input_ids=masked_input_ids,
            attention_mask=attention_mask,
            ss_input_ids=ss_input_ids,
        )

        log_probs = torch.log_softmax(
            outputs.logits[0, token_index],
            dim=-1,
        )

        wt_id = prosst_tokenizer.convert_tokens_to_ids(wt_used)
        mt_id = prosst_tokenizer.convert_tokens_to_ids(mt)

        mutation_score = log_probs[mt_id] - log_probs[wt_id]

        total_score += float(mutation_score.item())
        ref_positions.append(int(pos_ref))

    return float(total_score), bool(all_wt_match), ref_positions, len(muts)

In [ ]:
def calculate_spearman_results(
    df: pd.DataFrame,
    score_cols: list[str],
    prediction_col: str = "prosst_delta_logprob",
) -> pd.DataFrame:
    """
    Calculates Spearman correlations between ProSST scores and experimental columns.
    """
    rows = []

    x = pd.to_numeric(df[prediction_col], errors="coerce")

    for col in score_cols:
        if col not in df.columns:
            rows.append(
                {
                    "score_col": col,
                    "n": 0,
                    "spearman_rho": float("nan"),
                    "p_value": float("nan"),
                    "note": "missing column",
                }
            )
            continue

        y = pd.to_numeric(df[col], errors="coerce")
        mask = x.notna() & y.notna()

        n = int(mask.sum())

        if n < 3:
            rows.append(
                {
                    "score_col": col,
                    "n": n,
                    "spearman_rho": float("nan"),
                    "p_value": float("nan"),
                    "note": "not enough valid pairs",
                }
            )
            continue

        rho, p = spearmanr(
            x[mask].values,
            y[mask].values,
        )

        rows.append(
            {
                "score_col": col,
                "n": n,
                "spearman_rho": float(rho),
                "p_value": float(p),
                "note": "",
            }
        )

    return pd.DataFrame(rows)

In [ ]:
def run_dataset(cfg: DatasetConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs the complete ProSST zero-shot scoring pipeline for one dataset.
    This version keeps only multi-mutants.
    """
    print(f"\n==============================")
    print(f"Running {cfg.name}")
    print(f"==============================")

    df_raw = read_table(cfg)

    required_cols = [cfg.sequence_col, cfg.variant_col]
    missing_cols = [col for col in required_cols if col not in df_raw.columns]

    if missing_cols:
        raise ValueError(
            f"{cfg.name}: missing required columns: {missing_cols}"
        )

    df = df_raw.copy()

    # Optional: require num_mutations column and keep only multi-mutants
    if cfg.require_num_mutations_col:
        if "num_mutations" not in df.columns:
            raise ValueError(
                f"{cfg.name}: 'num_mutations' column is required but missing."
            )

        print("num_mutations distribution before filtering:")
        print(df["num_mutations"].value_counts().sort_index())

        df = df.loc[df["num_mutations"] > 1].copy()

        print("num_mutations distribution after multi-mutant filtering:")
        print(df["num_mutations"].value_counts().sort_index())

    # General valid-variant filter based on variant string
    df["prosst_is_valid_variant"] = df[cfg.variant_col].apply(is_valid_variant)
    df["prosst_num_mutations_parsed"] = df[cfg.variant_col].apply(count_mutations_from_variant)

    n_before = len(df)
    df = df.loc[df["prosst_is_valid_variant"]].reset_index(drop=True)
    n_after = len(df)

    print(f"Valid variants: {n_after}/{n_before}")
    print("Parsed mutation count distribution:")
    print(df["prosst_num_mutations_parsed"].value_counts().sort_index())

    if len(df) == 0:
        raise ValueError(
            f"{cfg.name}: no valid variant rows left after filtering."
        )

    wt_seq = str(df[cfg.sequence_col].iloc[0]).strip().upper()

    pdb_seq, structure_sequence = get_structure_from_pdb(cfg.pdb_path)

    validate_sequence_length(
        wt_seq=wt_seq,
        pdb_seq=pdb_seq,
        cfg=cfg,
    )

    # ProSST reference sequence comes from the PDB
    ref_seq = pdb_seq

    input_ids, attention_mask, ss_input_ids = tokenize_reference_sequence(
        ref_seq=ref_seq,
        structure_sequence=structure_sequence,
    )

    best_offset, hits, total = find_best_offset(
        df=df,
        ref_seq=ref_seq,
        variant_col=cfg.variant_col,
        search=cfg.offset_search,
    )

    match_rate = hits / max(total, 1)

    print(
        f"Offset used: {best_offset} | "
        f"WT matches: {hits}/{total} ({match_rate:.1%})"
    )

    scores = []
    wt_matches = []
    ref_positions = []
    num_mutations_scored = []

    for variant in tqdm(
        df[cfg.variant_col].tolist(),
        desc=f"Scoring {cfg.name}",
        unit="variant",
    ):
        score, matched, pos_refs, n_muts = score_variant(
            variant=variant,
            ref_seq=ref_seq,
            input_ids=input_ids,
            attention_mask=attention_mask,
            ss_input_ids=ss_input_ids,
            offset=best_offset,
            strict_wt_match=cfg.strict_wt_match,
        )

        scores.append(score)
        wt_matches.append(matched)
        ref_positions.append(",".join(map(str, pos_refs)))
        num_mutations_scored.append(n_muts)

    df_out = df.copy()

    df_out["prosst"] = scores
    df_out["prosst_delta_logprob"] = scores
    df_out["prosst_reference_sequence"] = "pdb_aa_seq"
    df_out["prosst_pdb_path"] = str(cfg.pdb_path)
    df_out["prosst_offset_used"] = best_offset
    df_out["prosst_pos_in_ref_1based"] = ref_positions
    df_out["prosst_num_mutations_scored"] = num_mutations_scored
    df_out["prosst_wt_match_after_offset"] = wt_matches

    save_table(df_out, cfg.out_path)

    print(f"Saved results: {cfg.out_path}")

    metrics = calculate_spearman_results(
        df=df_out,
        score_cols=cfg.score_cols,
        prediction_col="prosst_delta_logprob",
    )

    metrics.insert(0, "dataset", cfg.name)
    metrics.insert(1, "n_rows_scored", len(df_out))
    metrics.insert(2, "offset_used", best_offset)
    metrics.insert(3, "wt_match_rate", match_rate)

    print("\nSpearman results:")
    print(metrics.to_string(index=False))

    return df_out, metrics

In [ ]:
# Run All Datasets

all_outputs = {}
all_metrics = []

for cfg in DATASETS:
    df_out, metrics = run_dataset(cfg)

    all_outputs[cfg.name] = df_out
    all_metrics.append(metrics)

summary = pd.concat(all_metrics, ignore_index=True)

summary_path = OUT_DIR / "prosst_multi_spearman_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"\nSaved summary: {summary_path}")

summary


Running UBE4B_multi
num_mutations distribution before filtering:
num_mutations
1       940
2     52131
3     26572
4      6977
5      1454
6       256
7        35
8         8
9         1
10        1
Name: count, dtype: int64
Valid variants: 88375/88375
Parsed mutation count distribution:
prosst_num_mutations_parsed
1       940
2     52131
3     26572
4      6977
5      1454
6       256
7        35
8         8
9         1
10        1
Name: count, dtype: int64
---------- Building Subgraphs ----------


100%|██████████| 1/1 [00:01<00:00,  1.59s/it]


Offset used: 1 | WT matches: 221960/221960 (100.0%)


Scoring UBE4B_multi:   0%|          | 0/88375 [00:00<?, ?variant/s]

Saved results: /home/kebau102/Masterarbeit/out/prosst/multi/UBE4B_prosst_multi.tsv

Spearman results:
    dataset  n_rows_scored  offset_used  wt_match_rate score_col     n  spearman_rho  p_value note
UBE4B_multi          88375            1            1.0     score 88375       0.45372      0.0     

Running GRB2_multi
num_mutations distribution before filtering:
num_mutations
1     1034
2    62332
Name: count, dtype: int64
Valid variants: 63366/63366
Parsed mutation count distribution:
prosst_num_mutations_parsed
1     1034
2    62332
Name: count, dtype: int64
---------- Building Subgraphs ----------


...

100%|██████████| 1/1 [00:00<00:00,  1.83it/s]


Offset used: 1 | WT matches: 15061/15061 (100.0%)


Scoring DLG4_binding_multi:   0%|          | 0/8251 [00:00<?, ?variant/s]

Saved results: /home/kebau102/Masterarbeit/out/prosst/multi/DLG4_binding_prosst_multi.tsv

Spearman results:
           dataset  n_rows_scored  offset_used  wt_match_rate score_col    n  spearman_rho  p_value note
DLG4_binding_multi           8251            1            1.0     score 8251      0.806472      0.0     

Saved summary: /home/kebau102/Masterarbeit/out/prosst/multi/prosst_multi_spearman_summary.csv


,dataset,n_rows_scored,offset_used,wt_match_rate,score_col,n,spearman_rho,p_value,note
0,UBE4B_multi,88375,1,1.0,score,88375,0.453720,0.0,
1,GRB2_multi,63366,1,1.0,score,63366,0.727823,0.0,
2,DLG4_abundance_multi,6976,1,1.0,score,6976,0.784507,0.0,
3,DLG4_binding_multi,8251,1,1.0,score,8251,0.806472,0.0,
